### Task 1.

Оцените эксперимент «Sending email (correct link)» с использованием CUPED. В качестве ковариаты используйте выручку пользователей за 4 недели до эксперимента.

Данные эксперимента «Sending email (correct link)»: 2022-05-03/2022-05-03T12_df_sales.csv и 2022-05-03/experiment_users.csv. Эксперимент проводился с 2022-04-25 по 2022-05-02 (не включая!). Метрика — средняя выручка с клиента.

В качестве ответа введите p-value с точность до 4-го знака после точки.

**Решение**

In [1]:
from datetime import datetime, timedelta
import pandas as pd
import numpy as np
from scipy import stats

In [2]:
# загрузка данных
df_sales = pd.read_csv('2022-05-03T12_df_sales.csv')
df_sales['date'] = pd.to_datetime(df_sales['date'])

users = pd.read_csv('experiment_users.csv')

In [3]:
df_sales.head()

,sale_id,date,count_pizza,count_drink,price,user_id
0,1000001,2022-02-04 10:00:24,1,0,720,1c1543
1,1000002,2022-02-04 10:02:28,1,1,930,a9a6e8
2,1000003,2022-02-04 10:02:35,3,1,1980,23420a
3,1000004,2022-02-04 10:03:06,1,1,750,3e8ed5
4,1000005,2022-02-04 10:03:23,1,1,870,cbc468


Вычислим значения метрики и ковариат: стоимость покупок за период эксперимента и за 4 недели до эксперимента.

In [11]:
begin_exp_date = datetime(2022, 4, 25)
end_exp_date = datetime(2022, 5, 2)

df_metrics = (
    df_sales
    [(df_sales['date'] >= begin_exp_date) & (df_sales['date'] < end_exp_date)]
    .groupby('user_id')[['price']].sum()
    .rename(columns={'price': 'metric'})
)

begin_cov_date = begin_exp_date - timedelta(days=28)
df_cov = (
df_sales
[(df_sales['date'] >= begin_cov_date) & (df_sales['date'] < begin_exp_date)]
.groupby('user_id')[['price']].sum()
.rename(columns={'price': f'cov_4week'})
)
df_metrics = pd.merge(
  df_metrics,
  df_cov,
  left_index=True,
  right_index=True,
  how='outer'
).fillna(0).reset_index()

df_metrics.head(3)

,user_id,metric,cov_4week
0,0000d4,0.0,720.0
1,0000de,0.0,1320.0
2,0000e4,840.0,0.0


In [13]:
# выделяем выборки с пользовотелями
control_users = users[users.pilot == 0]
pilot_users = users[users.pilot == 1]

In [14]:
df_control = control_users.merge(df_metrics, on='user_id', how='left').fillna(0)
df_pilot = pilot_users.merge(df_metrics, on='user_id', how='left').fillna(0)

In [15]:
def calculate_theta(y_control, y_pilot, x_control, x_pilot):
    """Вычисляем Theta по данным двух групп.

    y_control - значения метрики во время пилота на контрольной группе
    y_pilot - значения метрики во время пилота на пилотной группе
    x_control - значения ковариант на контрольной группе
    x_pilot - значения ковариант на пилотной группе
    """
    y = np.hstack([y_control, y_pilot])
    x = np.hstack([x_control, x_pilot])
    covariance = np.cov(x, y)[0, 1]
    variance = x.var()
    theta = covariance / variance
    return theta

def check_cuped_test(df_control, df_pilot, covariate_column):
    """Проверяет гипотезу о равенстве средних с использованием CUPED.

    covariate_column - название стобца с ковариантой

    return - pvalue.
    """
    theta = calculate_theta(
        df_control['metric'], df_pilot['metric'],
        df_control[covariate_column], df_pilot[covariate_column]
    )
    metric_cuped_control = df_control['metric'] - theta * df_control[covariate_column]
    metric_cuped_pilot = df_pilot['metric'] - theta * df_pilot[covariate_column]
    _, pvalue = stats.ttest_ind(metric_cuped_control, metric_cuped_pilot)
    return pvalue

In [16]:
pvalue = check_cuped_test(df_control, df_pilot, 'cov_4week')
print(f'pvalue с CUPED: {pvalue:0.4f}')

pvalue с CUPED: 0.0539


**Ответ**: 0.0539

### Task 2.

Реализуйте функцию calculate_cuped_metric.

Обратите внимание, что в этом задании нужно использовать формулу из лекции с вычитанием среднего значения ковариаты.

Внимание!

Для вычисления параметра θ нужно оценить ковариацию и дисперсию. Оценивать их можно разными способами, например, есть смещённые и несмещённые оценки. На выборках маленького размера значения разных оценок могут сильно отличаться. Для решения этого задания используйте функции из библиотеки numpy с параметрами по умолчанию.

**Шаблон решения**

In [ ]:
import numpy as np
import pandas as pd


def calculate_cuped_metric(df_metric, df_cov):
    """Считает значения cuped-метрики.

    :param df_metric (pd.DataFrame): таблица со значениями метрики во время эксперимента
        со столбцами ['user_id', 'metric'].
    :param df_cov (pd.DataFrame): таблица со значениями ковариаты
        со столбцами ['user_id', 'cov'].
    :return df: таблица со значениями cuped-метрики со столбцами ['user_id', 'metric'].
    """
    # YOUR_CODE_HERE

**Пример**

In [ ]:
df_metric = pd.DataFrame({'user_id': [1, 2, 3], 'metric': [2000, 2500, 3000]})
df_cov = pd.DataFrame({'user_id': [1, 2, 3], 'cov': [1100, 1500, 0]})
df = calculate_cuped_metric(df_metric, df_cov)
# df = pd.DataFrame({'user_id': [1, 2, 3], 'metric': [2159.53, 2933.01, 2407.46]})

**Решение**

In [55]:
import numpy as np
import pandas as pd


def calculate_cuped_metric(df_metric, df_cov):
    """Считает значения cuped-метрики.

    :param df_metric (pd.DataFrame): таблица со значениями метрики во время эксперимента
        со столбцами ['user_id', 'metric'].
    :param df_cov (pd.DataFrame): таблица со значениями ковариаты
        со столбцами ['user_id', 'cov'].
    :return df: таблица со значениями cuped-метрики со столбцами ['user_id', 'metric'].
    """
    df_merged = df_metric.merge(df_cov, on='user_id', how='outer').fillna(0)
    covariance = np.cov(df_merged['cov'], df_merged['metric'])[0, 1]
    variance = np.var(df_merged['cov'])
    theta = covariance / variance
    df_merged['result'] = df_merged.metric - theta * (df_merged['cov'] - df_merged['cov'].mean())
    return df_merged[['user_id', 'result']].rename(columns={"result": "metric"})

In [56]:
df_metric = pd.DataFrame({'user_id': [1, 2, 3], 'metric': [2000, 2500, 3000]})
df_cov = pd.DataFrame({'user_id': [1, 2, 3], 'cov': [1100, 1500, 0]})
df = calculate_cuped_metric(df_metric, df_cov)
df
# df = pd.DataFrame({'user_id': [1, 2, 3], 'metric': [2159.53, 2933.01, 2407.46]})

,user_id,metric
0,1,2159.530387
1,2,2933.011050
2,3,2407.458564
